In [ ]:
# Notebook imports
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [ ]:
# Notebook configuration
FEATURES = [
    "spread_pct",
    "vwap_deviation",
    "return_1h",
    "log_return",
    "high_low_range",
    "vol_6h",
    "vol_24h",
]
WINDOW_Z = 30
Z_THRESHOLD = 3
PLOT_COLS = [
    "return_1h", "log_return", "vol_24h", "spread_pct", "vwap_deviation", "high_low_range"
 ]

COLORS = {
    "price": "#2563eb",
    "anomaly": "#dc2626",
    "normal": "#6b7280",
}

In [ ]:
# State check: requires engineered features
required = ["df_model", "FEATURES"]
missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError(
        "Missing prerequisites: " + ", ".join(missing) +
        ". Run 1.2-abz-features.ipynb first in the same kernel."
    )

missing_feats = [c for c in FEATURES if c not in df_model.columns]
if missing_feats:
    raise RuntimeError("df_model missing feature columns: " + ", ".join(missing_feats))
if "quote_datetime" not in df_model.columns:
    raise RuntimeError("df_model missing 'quote_datetime'. Run feature engineering first.")
print("Prerequisites OK. Proceed with statistical modeling.")


---
## 14. Initial Outlier Detection

Before building ML models, we look at simple statistical outliers
(returns beyond 3 standard deviations from the mean).

In [ ]:
# Features to analyze for outliers
features = FEATURES

# Calculate Multi-feature Z-Score Space using Rolling Statistics
window = WINDOW_Z
for col in features:
    if col in df_model.columns:
        feat_mean = df_model[col].rolling(window).mean()
        feat_std = df_model[col].rolling(window).std()
        df_model[f"{col}_z"] = (df_model[col] - feat_mean) / (feat_std + 1e-8)


z_threshold = Z_THRESHOLD
df_model['z_score'] = df_model['log_return_z']

df_model['is_outlier'] = (df_model['z_score'].abs() >= z_threshold).astype(int)
outliers = df_model[df_model['is_outlier'] == 1]

# Count total outliers
outlier_count = df_model['is_outlier'].sum()
print(f"Total Outliers Found: {outlier_count}")

# Get top 10 anomalies by spread_pct
top_10_hybrid = outliers.sort_values(by='spread_pct', ascending=False).head(10)

print("\nTop 10 Outliers")
display(top_10_hybrid[["quote_datetime", "close", "log_return", "z_score", "spread_pct"]])

Total Outliers Found: 256

Top 10 Outliers


,quote_datetime,close,log_return,z_score,spread_pct
13778,2025-01-30 10:30:00,415.2299,-0.061957,-4.334133,0.048178
14117,2025-04-09 13:30:00,379.0600,0.041228,3.112713,0.042201
13631,2024-12-27 10:30:00,428.3400,-0.022553,-3.639100,0.039692
14741,2025-10-28 10:30:00,545.6100,0.026371,3.832714,0.036658
13708,2025-01-15 10:30:00,425.2750,0.022820,3.182397,0.035267
13666,2025-01-06 10:30:00,432.6485,0.021679,3.530910,0.034671
13522,2024-12-04 10:30:00,438.1200,0.016060,3.166408,0.034243
14107,2025-04-08 10:30:00,372.1200,0.039382,3.899196,0.032248
14065,2025-03-31 10:30:00,367.8340,-0.029429,-4.063560,0.029908
14755,2025-10-30 10:30:00,528.3900,-0.026464,-3.261405,0.026493


In [ ]:
# Box Plots for primary features
fig = make_subplots(rows=2, cols=3, subplot_titles=[f'Box Plot: {col}' for col in PLOT_COLS])

for i, col in enumerate(PLOT_COLS):
    row = (i // 3) + 1
    col_idx = (i % 3) + 1
    fig.add_trace(
        go.Box(y=df_model[col], name=col, marker_color=COLORS['price'], boxpoints='outliers'),
        row=row, col=col_idx
    )

fig.update_layout(height=800, title_text="Statistical Feature Outliers", template="plotly_white", showlegend=False)
fig.show()

In [ ]:
# Inter-Quartile Range (IQR)

if "df_model" not in globals():
    raise ValueError("df_model not found. Run feature-engineering cell first.")

if "features" not in globals():
    features = ["spread_pct", "vwap_deviation", "return_1h", "log_return", "high_low_range", "vol_6h", "vol_24h"]

if "COLORS" not in globals():
    COLORS = {
        "price": "#2563eb",
        "anomaly": "#dc2626",
        "normal": "#6b7280"
    }

part = df_model.copy().sort_values("quote_datetime").reset_index(drop=True)
active_features = [c for c in features if c in part.columns]
window_iqr = 30
roll_window = 20

for col in active_features:
    q1 = part[col].rolling(window_iqr).quantile(0.25)
    q3 = part[col].rolling(window_iqr).quantile(0.75)
    iqr = q3 - q1
    part[f"{col}_iqr_flag"] = ((part[col] < (q1 - 1.5 * iqr)) | (part[col] > (q3 + 1.5 * iqr))).astype(int)

iqr_cols = [f"{c}_iqr_flag" for c in active_features if f"{c}_iqr_flag" in part.columns]
part["iqr_agg"] = part[iqr_cols].sum(axis=1)
part["iqr_flag"] = (part["iqr_agg"] > 0).astype(int)

mu = part["iqr_agg"].rolling(roll_window).mean()
sd = part["iqr_agg"].rolling(roll_window).std()
part["iqr_z20"] = (part["iqr_agg"] - mu) / (sd + 1e-8)

flagged = part[part["iqr_flag"] == 1]

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.05,
    subplot_titles=("Price with Outliers", "IQR Score")
)

fig.add_trace(go.Scatter(
    x=part["quote_datetime"], y=part["close"], name="Close Price",
    line=dict(color=COLORS["normal"], width=1), opacity=0.5
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=flagged["quote_datetime"], y=flagged["close"], mode="markers", name="Outliers",
    marker=dict(color=COLORS["anomaly"], size=6)
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=part["quote_datetime"], y=part["iqr_z20"], name="IQR Score",
    line=dict(color=COLORS["price"], width=0.8)
), row=2, col=1)

fig.add_hline(y=3, line_dash="dash", line_color="red", row=2, col=1)
fig.add_hline(y=-3, line_dash="dash", line_color="red", row=2, col=1)

fig.update_layout(height=800, title_text="IQR Outlier Detection Dashboard", template="plotly_white", showlegend=True)
fig.show()

print("IQR anomalies:", int(part["iqr_flag"].sum()))

IQR anomalies: 4888


In [ ]:
# Z-Score

if "df_model" not in globals():
    raise ValueError("df_model not found. Run feature-engineering cell first.")

features = FEATURES
part = df_model.copy().sort_values("quote_datetime").reset_index(drop=True)
active_features = [c for c in features if c in part.columns]
window_z = WINDOW_Z
roll_window = ROLL_WINDOW

for col in active_features:
    mu = part[col].rolling(window_z).mean()
    sd = part[col].rolling(window_z).std()
    part[f"{col}_z"] = (part[col] - mu) / (sd + 1e-8)

if "log_return_z" in part.columns:
    part["z_signal"] = part["log_return_z"]
else:
    z_cols = [f"{c}_z" for c in active_features if f"{c}_z" in part.columns]
    part["z_signal"] = part[z_cols].abs().max(axis=1)

part["z_flag"] = (part["z_signal"].abs() >= 3).astype(int)

mu2 = part["z_signal"].rolling(roll_window).mean()
sd2 = part["z_signal"].rolling(roll_window).std()
part["z_z20"] = (part["z_signal"] - mu2) / (sd2 + 1e-8)

flagged = part[part["z_flag"] == 1]

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.05,
    subplot_titles=("Price with Outliers", "Z Score")
)

fig.add_trace(
    go.Scatter(
        x=part["quote_datetime"],
        y=part["close"],
        name="Close Price",
        line=dict(color=COLORS["normal"], width=1),
        opacity=0.5,
    ),
    row=1, col=1,
 )

fig.add_trace(
    go.Scatter(
        x=flagged["quote_datetime"],
        y=flagged["close"],
        mode="markers",
        name="Outliers",
        marker=dict(color=COLORS["anomaly"], size=6),
    ),
    row=1, col=1,
 )

fig.add_trace(
    go.Scatter(
        x=part["quote_datetime"],
        y=part["z_z20"],
        name="Z-Score",
        line=dict(color=COLORS["price"], width=0.8),
    ),
    row=2, col=1,
 )

fig.add_hline(y=3, line_dash="dash", line_color="red", row=2, col=1)
fig.add_hline(y=-3, line_dash="dash", line_color="red", row=2, col=1)

fig.update_layout(height=800, title_text="Z-Score Outlier Detection Dashboard", template="plotly_white", showlegend=True)
fig.show()

print("Z Score anomalies:", int(part["z_flag"].sum()))